# NVE sweep at constant volume rate

This is the single-axis counterpart to `NVE_Sliding.ipynb`. Only one box length is changed while the other two remain fixed. Because $V=L_xL_yL_z$, making the selected length linear in time makes $dV/dt$ constant. Density is consequently *not* linear in time.

The first branch compresses to `rho_high`; the second expands back to the original density. Each pair is repeated for `number_of_cycles`, and the sweep is repeated for each requested number of steps per leg.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from md_Helpers import (
    ProjectPaths,
    SQLiteRunDatabase,
    open_run,
    run_clone_rescale_constant_volume_rate,
)

# --------------------------------------------------
# Configuration
# --------------------------------------------------
database = SQLiteRunDatabase(ProjectPaths().database)
original_run = "20260910171124"
nsteps_per_leg = [2_500, 5_000, 10_000, 25_000]
number_of_cycles = 3
rho_high = 0.7
resize_axis = "x"  # May be "x", "y", or "z"
density_grid_points = 250
smoothing_window = 1

records = database.query_thermalizations(Run_ID=original_run)
if not records:
    raise ValueError(f"Run {original_run} was not found")
rho_original = float(records[0]["Density_End"])
target_densities = [rho_high, rho_original]
number_of_legs = 2 * number_of_cycles

sweep_runs = {}
for nsteps in nsteps_per_leg:
    print(f"\nRunning {nsteps:,} steps per leg")
    source_run_id = original_run
    run_ids = []

    for leg_index in range(number_of_legs):
        target_density = target_densities[leg_index % 2]
        direction = "compression" if target_density > rho_original else "expansion"
        result = run_clone_rescale_constant_volume_rate(
            source_run_id=source_run_id,
            final_density=target_density,
            nsteps=nsteps,
            axis=resize_axis,
            ensemble="NVE",
            notes=(
                f"Constant-dV/dt NVE triangle; axis={resize_axis}; "
                f"Nsteps={nsteps}; cycle={leg_index // 2 + 1}; "
                f"direction={direction}"
            ),
        )
        source_run_id = result["run_id"]
        run_ids.append(source_run_id)
        print(f"  Leg {leg_index + 1}: {direction}, Run_ID={source_run_id}")

    sweep_runs[nsteps] = run_ids


In [ ]:
# Verify that volume is linear in time for every generated leg.
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
volume_linearity = {}

for ax, nsteps in zip(axes.flat, nsteps_per_leg):
    maximum_relative_residual = 0.0
    for leg_index, run_id in enumerate(sweep_runs[nsteps], start=1):
        logs = open_run(run_id).logs_dataframe()
        fraction = logs["run_step"].to_numpy(dtype=float) / nsteps
        volume = logs["volume"].to_numpy(dtype=float)
        expected = volume[0] + fraction * (volume[-1] - volume[0])
        scale = max(abs(volume[-1] - volume[0]), np.finfo(float).eps)
        residual = (volume - expected) / scale
        maximum_relative_residual = max(
            maximum_relative_residual,
            float(np.max(np.abs(residual))),
        )
        ax.plot(fraction, residual, linewidth=1.2, label=f"Leg {leg_index}")

    volume_linearity[nsteps] = maximum_relative_residual
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"{nsteps:,} steps; max |residual| = {maximum_relative_residual:.2e}")
    ax.set_xlabel("Fraction of leg")
    ax.set_ylabel(r"$(V-V_{linear})/|\Delta V|$")
    ax.grid(alpha=0.3)

fig.suptitle(f"Constant-volume-rate check: only {resize_axis}-length changes")
plt.tight_layout()
plt.show()
volume_linearity


In [ ]:
# Average the repeated compression and expansion branches, then plot their residual.
averaged_curves = {}
for nsteps in nsteps_per_leg:
    branches = {"compression": [], "expansion": []}
    for run_id in sweep_runs[nsteps]:
        logs = open_run(run_id).logs_dataframe()
        density = logs["density"].to_numpy(dtype=float)
        pressure = logs["pressure"].to_numpy(dtype=float)
        direction = "compression" if density[-1] > density[0] else "expansion"
        branches[direction].append((density, pressure))

    averaged_curves[nsteps] = {}
    for direction, runs in branches.items():
        if len(runs) != number_of_cycles:
            raise RuntimeError(
                f"Expected {number_of_cycles} {direction} legs for {nsteps:,} steps; "
                f"found {len(runs)}"
            )
        rho_min = max(density.min() for density, _ in runs)
        rho_max = min(density.max() for density, _ in runs)
        grid = np.linspace(rho_min, rho_max, density_grid_points)
        pressure_grid = np.asarray([
            np.interp(grid, density[np.argsort(density)], pressure[np.argsort(density)])
            for density, pressure in runs
        ])
        mean = pressure_grid.mean(axis=0)
        smoothed = (
            pd.Series(mean)
            .rolling(smoothing_window, center=True, min_periods=1)
            .mean()
            .to_numpy()
        )
        averaged_curves[nsteps][direction] = {
            "density": grid,
            "pressure": smoothed,
            "std": pressure_grid.std(axis=0, ddof=1),
        }

fig = plt.figure(figsize=(14, 12))
outer = fig.add_gridspec(2, 2, hspace=0.30, wspace=0.18)
styles = {
    "compression": ("tab:blue", "Mean compression"),
    "expansion": ("tab:orange", "Mean expansion"),
}

for panel_index, nsteps in enumerate(nsteps_per_leg):
    row, column = divmod(panel_index, 2)
    inner = outer[row, column].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.05)
    ax = fig.add_subplot(inner[0])
    ax_residual = fig.add_subplot(inner[1], sharex=ax)

    for direction in ("compression", "expansion"):
        curve = averaged_curves[nsteps][direction]
        color, label = styles[direction]
        ax.plot(curve["density"], curve["pressure"], color=color, lw=2.5, label=label)
        ax.fill_between(
            curve["density"],
            curve["pressure"] - curve["std"],
            curve["pressure"] + curve["std"],
            color=color, alpha=0.12,
        )

    compression = averaged_curves[nsteps]["compression"]
    expansion = averaged_curves[nsteps]["expansion"]
    rho_min = max(compression["density"].min(), expansion["density"].min())
    rho_max = min(compression["density"].max(), expansion["density"].max())
    grid = np.linspace(rho_min, rho_max, density_grid_points)
    residual = (
        np.interp(grid, compression["density"], compression["pressure"])
        - np.interp(grid, expansion["density"], expansion["pressure"])
    )
    ax_residual.plot(grid, residual, color="black", lw=1.8)
    ax_residual.axhline(0, color="gray", ls="--", lw=1)
    ax.set_title(f"Nsteps per leg = {nsteps:,}")
    ax.set_ylabel("Mean pressure")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", labelbottom=False)
    ax_residual.set_xlabel(r"Density, $\rho=N/V$")
    ax_residual.set_ylabel(r"$P_{comp}-P_{exp}$")
    ax_residual.grid(alpha=0.3)

fig.suptitle(f"NVE constant-dV/dt triangle (single {resize_axis}-axis ramp)", fontsize=14)
plt.show()
